In [0]:
from pyspark.sql.functions import (
    sum as _sum, count, avg, round,
    year, month, quarter, dayofweek,
    datediff, current_date, countDistinct,
    when, max as _max, min as _min
)

from pyspark.sql.functions import when, col

silver = spark.read.table("bi.silver.orders_denormalized")

# sprzedaż wg kraju klienta
gold_country = (
    silver.groupBy("customer_country")
    .agg(
        round(_sum("net_revenue"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders"),
        round(avg("net_revenue"), 2).alias("avg_order_value")
    )
    .orderBy("total_revenue", ascending=False)
)
gold_country.write.format("delta").mode("overwrite").saveAsTable("bi.gold.sales_by_country")

# sprzedaż wg kategorii produktu
gold_category = (
    silver.groupBy("category_name")
    .agg(
        round(_sum("net_revenue"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders"),
        round(avg("net_revenue"), 2).alias("avg_order_value")
    )
    .orderBy("total_revenue", ascending=False)
)
gold_category.write.format("delta").mode("overwrite").saveAsTable("bi.gold.sales_by_category")

# sprzedaż wg miesiąca
gold_monthly = (
    silver
    .groupBy(
        year("order_date").alias("year"),
        month("order_date").alias("month")
    )
    .agg(
        round(_sum("net_revenue"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
    .orderBy("year", "month")
)
gold_monthly.write.format("delta").mode("overwrite").saveAsTable("bi.gold.sales_monthly_trend")

# wyniki pracowników
gold_emp = (
    silver.groupBy("employee_name")
    .agg(
        round(_sum("net_revenue"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
    .orderBy("total_revenue", ascending=False)
)
gold_emp.write.format("delta").mode("overwrite").saveAsTable("bi.gold.employee_performance")


gold_customer_features = (
    silver
    .groupBy("customer_id", "customer_name", "customer_country", "customer_city")
    .agg(
        round(_sum("net_revenue"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders"),
        round(avg("net_revenue"), 2).alias("avg_order_value"),
        countDistinct("product_id").alias("unique_products"),
        countDistinct("category_name").alias("unique_categories"),
        round(avg("freight"), 2).alias("avg_freight"),
        _min("order_date").alias("first_order_date"),
        _max("order_date").alias("last_order_date")
    )
    .withColumn(
        "days_since_last_order",
        datediff(current_date(), col("last_order_date"))
    )
    .withColumn(
        "customer_lifetime_days",
        datediff(col("last_order_date"), col("first_order_date"))
    )
    .withColumn(
        "estimated_clv",
        round(col("avg_order_value") * col("total_orders"), 2)
    )
    .withColumn(
        "customer_segment",
        when(col("total_revenue") >= 10000, "High Value")
        .when(col("total_revenue") >= 4000, "Medium Value")
        .otherwise("Low Value")
    )
)

gold_customer_features.write.format("delta").mode("overwrite").saveAsTable("bi.gold.customer_features_ml")

print("[GOLD] Wszystkie tabele gold załadowane.")
print("[GOLD] Utworzono tabelę feature table: bi.gold.customer_features_ml")

In [0]:
from pyspark.sql.functions import (
    sum as _sum, count, avg, max as _max,
    round, datediff, to_date, lit, rank, desc

)
from pyspark.sql.window import Window

silver = spark.read.table("bi.silver.orders_denormalized")

#  klienci wg wartości
gold_customers = (
    silver
    .groupBy("customer_id", "customer_name", "customer_country", "customer_city")
    .agg(
        round(_sum("net_revenue"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders"),
        round(avg("net_revenue"), 2).alias("avg_order_value"),
        _max("order_date").alias("last_order_date")
    )
    .withColumn("revenue_rank",
        rank().over(Window.orderBy(
            __import__('pyspark.sql.functions', fromlist=['desc']).desc("total_revenue")
        ))
    )
    .orderBy(desc("total_revenue"))
)

gold_customers.write.format("delta").mode("overwrite") \
    .saveAsTable("bi.gold.customer_value")

# top 10 klientów
top10 = gold_customers.limit(10)
display(top10)


In [0]:
from pyspark.sql.functions import when, col
segmented = gold_customers.withColumn(
    "segment",
    when(col("total_revenue") >= 10000, "High Value")
    .when(col("total_revenue") >= 4000,  "Medium Value")
    .otherwise("Low Value")
)

segmented.write.format("delta").mode("overwrite") \
    .saveAsTable("bi.gold.customer_segments")

display(segmented.groupBy("segment").agg(
    count("customer_id").alias("liczba_klientow"),
    round(_sum("total_revenue"), 2).alias("suma_przychodu")
).orderBy("suma_przychodu", ascending=False))